# Chapter 16: Temperature and Top-k

[Read this chapter online](https://jackluu.io/book/section-5-generation/ch16-temperature-and-topk/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch16-temperature-and-topk.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 16: Temperature and Top-k

![You are here in the big picture](../assets/diagrams/ch16-where-we-are.png){ width="756" }
*Figure 16.1: Where we are: we are tuning the text generation process.*

Plain sampling gives us varied output, but it can be unpredictable. Sometimes the model chooses a very rare character that completely breaks the grammar or creates a nonsense word. We need controls to balance creativity with coherence. In this chapter you will:

- Use temperature to adjust how risky the model's choices are.
- Use top-k truncation to prevent the model from picking truly bad characters.
- Combine these controls for optimal text generation.

**Words to Know**
    - **Temperature**: a number that scales the logits before they are turned into probabilities, controlling the randomness of the output.
    - **Top-k**: a limit that restricts the model to only sample from the *k* most likely next tokens, ignoring all others.

## Theory

Two controls sit between the raw scores the model produces and the character it finally picks. Temperature reshapes the odds. Top-k decides which characters are allowed to compete at all. They are independent, and in practice you set both.

![The two generation controls side by side](../assets/diagrams/ch16-controls.png){ width="718" }
*Figure 16.2: Temperature changes the shape of the odds; top-k changes how many characters stay in the running.*

### Temperature

Temperature is a simple math trick applied to the logits before the softmax step. By dividing all the raw scores by a number (the temperature), we change the shape of the probability distribution.

```python
$ python src/examples/ch16_temperature.py
Saved chart: ch16_temperature.png
```

![Bars of next-character probabilities at three temperatures](../assets/plots/ch16_temperature.png){ width="650" }
*Figure 16.3: A plotted chart showing probabilities: low temperature sharpens the choice, high temperature flattens it.*

- **Low temperature (e.g., 0.5)**: Dividing by a fraction mathematically stretches the scores apart. The top choice becomes overwhelmingly favored. The model plays it safe.
- **High temperature (e.g., 2.0)**: Dividing by a large number mathematically squishes the scores together. The probabilities flatten out, meaning unusual choices become more likely. The model takes risks.
- **T = 1.0**: This is standard sampling. The model uses its learned probabilities exactly as they are.

### Top-k

Even at a safe temperature, there is a tiny mathematical chance (say, 0.001%) that the model might pick a completely absurd character. If you generate 200 characters, those tiny chances add up, and a mistake is likely.

Top-k sampling solves this by putting a hard limit on the choices. If `top_k = 40`, we look at the 65 possible characters, keep the 40 with the highest scores, and completely discard the bottom 25 by setting their probabilities to zero. This cuts off the "long tail" of bad choices, guaranteeing that the model only samples from the most reasonable options.

![Ranked scores with the low-scoring tail greyed out](../assets/diagrams/ch16-top-k.png){ width="619" }
*Figure 16.4: Top-k keeps the highest-scoring characters and zeroes the rest. Drawn here with ten characters and k = 4; the book's code uses 65 and k = 40.*

As the "where we are" map shows, these controls adjust the Next-Token Scores right before generation loops back to produce the New Text.

**In Business**
    For the house-style assistant, you might use a low temperature (0.5) when generating compliance documentation to ensure it stays close to the safest boilerplate text. When brainstorming marketing slogans, a higher temperature (0.9) with top-k sampling (40) will produce creative, surprising slogans that still make grammatical sense.

## Code

We apply temperature and top-k right before the softmax function in the generation loop.

```python
# Temperature controls randomness: lower is less random
        logits = logits / temperature

        # Top-k sampling limits the choices to the k most likely tokens
        if top_k is not None:
            threshold = logits.topk(top_k).values[:, -1, None]
            logits = logits.masked_fill(logits < threshold, float("-inf"))

        probs   = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
```

Line 2 divides the scores by the temperature to adjust randomness. Line 7 discards any score below the top-k threshold by setting it to negative infinity.

![Code flow: logits divided by temperature, filtered by top-k, then softmaxed](../assets/diagrams/ch16-code-flow.png){ width="738" }
*Figure 16.5: The code flow applies temperature first, then top-k, and finally softmax.*

In PyTorch, we use `masked_fill` to apply top-k. We find the score of the 40th best token (`threshold`), and any token with a score lower than that gets its value changed to negative infinity (`-inf`). When softmax calculates probabilities, anything with a score of `-inf` mathematically becomes exactly 0. 

Let's see how different settings affect the generated text.

```python
$ python src/ch15_generate_sampling.py
--- temp=0.5, top_k=40 ---
JULIET:
My must grace prove a son the come on my lord.

...
--- temp=0.8, top_k=40 ---
JULIET:
That will, my call thee for thee king win a be parder
The his in suppy ascented a not a say?
...
--- temp=1.0, top_k=40 ---
JULIET:
Look, do indo witned me devise thy grohed:
A give gring brothe whith ighers paison:
...
--- temp=1.5, top_k=40 ---
JULIET:
fattlyals give niet, thou stain, viful-wid thel,
Brob dayse mernion shing you fea,
...
--- temp=1.0, no top_k ---
JULIET:
Than mide smear of thou I am go vious, you me
now you scalf them frate ell, to 'till will she
...
--- Sweet spot ---
temperature=0.8 to 1.0 and top_k=40 usually gives the best results.
```

**What just happened:**

- At `temp=0.5`, the text is coherent but relies heavily on common, safe words.
- At `temp=0.8` to `1.0`, the text feels more natural and varied.
- At `temp=1.5`, the text quickly devolves into chaos and made-up words ("fattlyals").
- The sweet spot is typically a temperature between 0.8 and 1.0, combined with `top_k=40`.

### Shape Check

Table 16.1 outlines the shapes of tensors used during top-k filtering.

**Table 16.1:** Tensors modified during the top-k sampling process.

| Tensor | Shape | What it means |
|--------|-------|---------------|
| `threshold` | `[1, 1]` | The cutoff score (the 40th highest value). |
| `logits` | `[1, 65]` | 65 scores, where 25 of them have been set to `-inf`. |

## Try It

**Try It**
    Open the existing script `src/examples/ch16_explore_temp.py` and run it. It uses a very low temperature (`0.1`). You will notice it behaves almost exactly like greedy decoding, picking the safe top character every time!

    ```console title="Terminal"
    $ python src/examples/ch16_explore_temp.py
    --- Prompt: 'JULIET:\n' ---
    Generating with T=0.1...
    JULIET:
    I will the shall the shall be the son th...

    ```

## Key Takeaways

- Temperature adjusts the spread of the probabilities. Low temperature makes the model conservative, while high temperature makes it creative and risky.
- Top-k restricts the model from ever picking the worst-scoring tokens, preventing bizarre errors.
- To combine them, first apply temperature, then apply top-k to filter out the bad options, and finally convert to probabilities with softmax.
- A combination of `temperature=0.8` and `top_k=40` is a widely used default for high-quality text generation.

## Check Your Understanding

1. If you set the temperature to 0.1, what happens to the gap between the highest and lowest scores?
2. Why do we replace the eliminated scores in top-k with negative infinity (`-inf`) instead of `0`?
3. Which step must happen first in the code: applying top-k or calculating softmax probabilities?


## Further Reading

**Where top-k sampling comes from.** This paper is about writing stories from a prompt, and along the way it introduced the sampling rule you use in Chapter 16: keep only the k most likely next tokens and draw from those. It keeps the text varied without letting the model pick something absurd from the long tail.

**Why always picking the likeliest word goes wrong.** Always taking the highest-scoring next token, which Chapter 15 calls greedy decoding, produces flat and repetitive text, and this paper shows why: human writing is not made of the most predictable word at every turn. Their alternative keeps the smallest set of tokens whose probabilities add up to a chosen share and samples from that. How you choose the next token matters as much as how well the model was trained.

<div class="refs" markdown>

Fan, A., Lewis, M., & Dauphin, Y. (2018). *Hierarchical neural story generation* (arXiv:1805.04833). arXiv. https://doi.org/10.48550/arXiv.1805.04833

Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2019). *The curious case of neural text degeneration* (arXiv:1904.09751). arXiv. https://doi.org/10.48550/arXiv.1904.09751

</div>

---

### `src/ch15_generate_sampling.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch15_generate_sampling.py"   # a cell has none, and the file uses it to find the text

"""
Explore temperature and top-k sampling.
This file belongs to Chapter 16.
Run: python src/ch15_generate_sampling.py
"""
import os
import sys
import torch
import torch.nn.functional as F


from src.utils.config import GPTConfig
from src.ch09_gpt_model import GPT

# Settings
CHECKPOINT_PATH = "checkpoints/model.pt"
DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

# --- The Idea ---

def generate(
    model, prompt, encode, decode, cfg, max_new_tokens=200,
    temperature=1.0, top_k=None
):
    ids = torch.tensor([encode(prompt)], dtype=torch.long)

    for _ in range(max_new_tokens):
        ctx    = ids[:, -cfg.block_size:]
        logits = model(ctx)
        logits = logits[:, -1, :]

        # Temperature controls randomness: lower is less random
        logits = logits / temperature

        # Top-k sampling limits the choices to the k most likely tokens
        if top_k is not None:
            threshold = logits.topk(top_k).values[:, -1, None]
            logits = logits.masked_fill(logits < threshold, float("-inf"))

        probs   = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids     = torch.cat([ids, next_id], dim=1)

    return decode(ids[0].tolist())

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 16: Temperature and Top-k Sampling\n")

    if not os.path.exists(DATA_PATH):
        print("ERROR: Run: python src/utils/download_data.py")
        sys.exit(1)

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    chars  = sorted(set(text))
    char_to_id   = {ch: i for i, ch in enumerate(chars)}
    id_to_char   = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [char_to_id[c] for c in s if c in char_to_id]
    decode = lambda ids: "".join([id_to_char[i] for i in ids])

    checkpoint = torch.load(
        CHECKPOINT_PATH, map_location="cpu", weights_only=False
    )
    cfg   = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    print(f"Model loaded (trained {checkpoint['step']} steps)\n")

    prompt = "JULIET:\n"
    n_tokens = 150

    print(f"Prompt: {repr(prompt)}")
    print(f"Generating {n_tokens} characters each...\n")

    configs = [
        {"temperature": 0.5, "top_k": 40,   "label": "temp=0.5, top_k=40"},
        {"temperature": 0.8, "top_k": 40,   "label": "temp=0.8, top_k=40"},
        {"temperature": 1.0, "top_k": 40,   "label": "temp=1.0, top_k=40"},
        {"temperature": 1.5, "top_k": 40,   "label": "temp=1.5, top_k=40"},
        {"temperature": 1.0, "top_k": None, "label": "temp=1.0, no top_k"},
    ]

    with torch.no_grad():
        for cfg_dict in configs:
            text_out = generate(
                model, prompt, encode, decode, cfg,
                max_new_tokens=n_tokens,
                temperature=cfg_dict["temperature"],
                top_k=cfg_dict["top_k"]
            )
            print("---", cfg_dict["label"], "---")
            print(text_out)
            print()

    print("--- Sweet spot ---")
    print("temperature=0.8 to 1.0 and top_k=40 usually gives the best results.")

---

### `src/examples/ch16_explore_temp.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch16_explore_temp.py"   # a cell has none, and the file uses it to find the text

"""Generate text with a very low temperature to simulate greedy decoding."""
import os
import sys
import torch

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode, decode
from src.ch09_gpt_model import GPT
from src.ch15_generate_sampling import generate

def main():
    torch.manual_seed(42)
    checkpoint_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "checkpoints", "model.pt"
    )
    
    if not os.path.exists(checkpoint_path):
        print("Checkpoint not found.")
        return
        
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )
    
    cfg = checkpoint["gpt_cfg"]
    model = GPT(cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    
    text_path = os.path.join(
        os.path.dirname(__file__), "..", "..", "src", "data", "shakespeare.txt"
    )
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()
    chars, char_to_id, id_to_char = build_vocab(text)
    
    encode_fn = lambda s: encode(s, char_to_id)
    decode_fn = lambda ids: decode(ids, id_to_char)
    
    prompt = "JULIET:\n"
    print(f"--- Prompt: {repr(prompt)} ---")
    print("Generating with T=0.1...")
    
    # Using the generation function that takes temperature and top_k
    output = generate(
        model, prompt, encode_fn, decode_fn, cfg, 
        max_new_tokens=40, temperature=0.1, top_k=40
    )
    print(output + "...\n")

if __name__ == "__main__":
    main()

---

### `src/examples/ch16_temperature.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch16_temperature.py"   # a cell has none, and the file uses it to find the text

"""Plot the effect of temperature on a probability distribution."""
import matplotlib.pyplot as plt
import numpy as np

def main():
    # Example scores (logits)
    logits = np.array([2.5, 1.8, 0.5, -1.0, -2.5])
    labels = ["A", "B", "C", "D", "E"]
    
    temps = [0.5, 1.0, 2.0]
    
    fig, axes = plt.subplots(1, 3, figsize=(6.5, 3.5), dpi=300)
    fig.patch.set_facecolor('white')
    
    for ax, T in zip(axes, temps):
        # Apply temperature
        scaled = logits / T
        # Softmax
        exp_L = np.exp(scaled - np.max(scaled))
        probs = exp_L / np.sum(exp_L)
        
        ax.bar(labels, probs, color="#00695C")
        ax.set_ylim(0, 1.0)
        ax.set_title(f"T = {T}")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig("ch16_temperature.png")
    print("Saved chart: ch16_temperature.png")

if __name__ == "__main__":
    main()